# Borzoi Regulatory Sequence Prediction

![Borzoi Regulatory Sequence Prediction](https://proto-bio.github.io/proto-assets/images/tool/borzoi/hero.png)

Borzoi is a long-context regulatory sequence model published by Linder et al. (2024) that predicts regulatory signals from 524,288 bp DNA input windows. It extends the Enformer architecture with larger context, higher resolution output (6,144 bins), and replicate-specific checkpoints trained on different data splits. The tool provides two functions: single-replicate prediction for fast inference with one checkpoint, and ensemble prediction that runs all four replicates for robust consensus analysis with uncertainty estimates.

This notebook demonstrates both workflows, showing how to extract selected output tracks and compute cross-replicate statistics.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("borzoi")
display_overview("borzoi")
display_docs_section("borzoi", "Background")

# Borzoi

[Borzoi](https://github.com/calico/borzoi) is a deep learning model that predicts cell-type and tissue-specific RNA-seq coverage and other regulatory genomics tracks directly from DNA sequence, developed by the Kelley lab at [Calico Life Sciences](https://www.calicolabs.com/). Given a fixed 524,288 bp input window, it returns activity across 6,144 bins of 32 bp each for thousands of human or mouse genomic assays. This toolkit exposes single-replicate prediction and a four-replicate ensemble for uncertainty estimation.

Gene regulation acts across a wide range of genomic distances. Most promoter-proximal elements operate within a few kilobases, yet enhancers can influence genes from more than 100 kb away, and [topologically associating domains](https://en.wikipedia.org/wiki/Topologically_associating_domain) organize chromatin contacts over megabase scales. Sequence-to-function models that aim to relate noncoding variation to molecular phenotype therefore require an input window broad enough to capture these long-range relationships at fine spatial resolution.

Borzoi ([Linder et al., 2025](https://doi.org/10.1038/s41588-024-02053-6)) learns to predict cell- and tissue-specific [RNA-seq](https://en.wikipedia.org/wiki/RNA-Seq) coverage from DNA sequence, serving as a unifying model of gene regulation. Using statistics computed from its predicted coverage, Borzoi isolates and accurately scores the DNA regulatory elements that modulate transcriptional processes including transcription, splicing, and polyadenylation, with greater accuracy than comparable models. Benchmarked against state-of-the-art models, it accurately predicts the influence of variants on RNA expression and splicing and recapitulates the causal variants underlying molecular quantitative trait loci. Alongside RNA-seq, the model predicts [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression), [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq), [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq), and [histone modification](https://en.wikipedia.org/wiki/Histone_modification) tracks, making it a broad regulatory genomics predictor.

The published model couples a convolutional sequence encoder with transformer-style attention to process the full 524,288 bp window, and separate output heads produce human and mouse track predictions. The Borzoi authors trained four model replicates from independent initializations, which this toolkit exposes both as a single-replicate tool and as a four-replicate ensemble. A separate [FlashAttention](https://github.com/Dao-AILab/flash-attention)-based distillation of Borzoi, named Flashzoi, reaches comparable accuracy at substantially higher speed. The human checkpoints exposed by this toolkit use the Flashzoi distillation, and the mouse checkpoints use the standard Borzoi architecture.

## Available tools

In [2]:
display_available_tools("borzoi")

- **`run_borzoi_ensemble()`** — Regulatory activity prediction using all 4 Borzoi replicates
- **`run_borzoi()`** — Regulatory activity prediction using a single Borzoi replicate

### `run_borzoi`

This function runs a single Borzoi replicate checkpoint on a 524,288 bp DNA sequence and returns a prediction matrix of shape [num_tracks, 6144]. It is the fastest option for regulatory activity prediction when cross-replicate uncertainty estimates are not required. The replicate index (0 through 3) and species (human or mouse) can be configured independently, and output tracks can be averaged into a single signal for compact downstream analysis.

In [3]:
import numpy as np
from proto_tools import (
    BORZOI_CONTEXT,
    BorzoiConfig,
    BorzoiInput,
    run_borzoi,
)

In [4]:
# Display input docs
display_api_reference("borzoi", "input", "run_borzoi")

# Create a full-length DNA sequence by repeating a short pattern to fill the required context
sequence = "ATCG" * (BORZOI_CONTEXT // 4)
inputs = BorzoiInput(sequences=[sequence])

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[proto_tools.tools.sequence_scoring.shared_data_models.SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [5]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi")

single_config = BorzoiConfig(
    output_tracks=[0, 1, 2],
    species="human",
    replicate="0",
    avg_output_tracks=False,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Borzoi output tracks to extract (7611 human / 2608 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>replicate</code> | <code>Literal['0', '1', '2', '3']</code> | <code>'0'</code> | Replicate ID to run |
| <code>avg_output_tracks</code> | <code>bool</code> | <code>True</code> | Whether to average selected tracks into one output |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Number of sequences to process simultaneously on GPU |

In [6]:
# Run the single-replicate prediction function
single_result = run_borzoi(inputs, single_config)

Running run_borzoi [00:00]

In [7]:
display_api_reference("borzoi", "output", "run_borzoi")

print(f"Tracks: {len(single_result.results[0].prediction)}")
print(f"Bins per track: {len(single_result.results[0].prediction[0])}")

**Output** — `BorzoiOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.borzoi.borzoi_prediction.BorzoiPredictionResult]</code> | required | Per-sequence Borzoi prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices used for prediction |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |
| <code>replicate</code> | <code>str</code> | required | Replicate used for prediction ('0' to '3') |
| <code>avg_output_tracks</code> | <code>bool</code> | required | Whether track outputs were averaged |

Tracks: 3
Bins per track: 6144


### `run_borzoi_ensemble`

This function runs all four Borzoi replicate checkpoints and stacks the outputs into a tensor of shape [4, num_tracks, 6144]. Access to cross-replicate mean and standard deviation provides a measure of prediction confidence: regions where replicates agree strongly (low standard deviation) are more likely to reflect genuine regulatory signal than regions with high variance. When `avg_output_tracks` is enabled, each replicate's tracks are averaged before stacking, giving a final shape of [4, 1, 6144].

In [8]:
from proto_tools import (
    BorzoiEnsembleConfig,
    run_borzoi_ensemble,
)

In [9]:
# Display input docs (same BorzoiInput as single-replicate prediction)
display_api_reference("borzoi", "input", "run_borzoi_ensemble")

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[proto_tools.tools.sequence_scoring.shared_data_models.SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [10]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi_ensemble")

ensemble_config = BorzoiEnsembleConfig(
    output_tracks=[0, 1, 2],
    species="human",
    avg_output_tracks=True,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiEnsembleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Borzoi output tracks to extract (7611 human / 2608 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>avg_output_tracks</code> | <code>bool</code> | <code>True</code> | Whether to average selected tracks into one output |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Number of sequences to process simultaneously on GPU |

In [11]:
# Run the ensemble prediction function (runs all 4 replicates)
ensemble_result = run_borzoi_ensemble(inputs, ensemble_config)

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

In [12]:
display_api_reference("borzoi", "output", "run_borzoi_ensemble")

ensemble_array = np.array(ensemble_result.results[0].predictions)
mean_pred = ensemble_array.mean(axis=0)
std_pred = ensemble_array.std(axis=0)
print(f"Replicates: {ensemble_array.shape[0]}")
print(f"Mean shape: {mean_pred.shape}")
print(f"Std max: {std_pred.max():.4f}")

**Output** — `BorzoiEnsembleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[proto_tools.tools.sequence_scoring.borzoi.borzoi_ensemble.BorzoiEnsemblePredictionResult]</code> | required | Per-sequence ensemble prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices used for prediction |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |
| <code>avg_output_tracks</code> | <code>bool</code> | required | Whether track outputs were averaged |
| <code>num_replicates</code> | <code>int</code> | <code>4</code> | Number of Borzoi replicates returned |

Replicates: 4
Mean shape: (1, 6144)
Std max: 0.0051


## Export Results

Single-replicate and ensemble outputs can be saved to standard file formats for downstream analysis. Supported formats include JSON and CSV.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

single_result.export(name="borzoi_single", export_path=output_dir, file_format="json")
print(f"Single-replicate result exported to {output_dir}/borzoi_single.json")

ensemble_result.export(name="borzoi_ensemble", export_path=output_dir, file_format="csv")
print(f"Ensemble summary exported to {output_dir}/borzoi_ensemble.csv")

Single-replicate result exported to example_output/borzoi_single.json
Ensemble summary exported to example_output/borzoi_ensemble.csv
